# 04 — Decision Support

## tl;dr

The analysis flags **37 single-source parts**, **40 potential stockouts**, and **13 parts requiring immediate mitigation**. Of those actions, 11 are `Expedite and allocate` and 2 are `Qualify a second source`. Recommendations are deterministic responses to visible conditions—not automated purchasing decisions.

## Context & Methods

This notebook asks: **How should management translate supplier and inventory risk into an ordered action queue?** It combines part criticality, sourcing concentration, supplier risk, days of supply, and replenishment lead time.

### Key Assumptions

- Potential stockout means days of supply are below average historical replenishment lead time.
- Single-source status is based only on the suppliers present in the synthetic part map.
- Actions are triage rules and require buyer or planner review before execution.

### 1. Load part-level decision data

In [1]:
from pathlib import Path
import pandas as pd

from src.supply_chain_risk.analytics import RISK_WEIGHTS, apply_weight_scenario, build_analysis, load_data

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
tables, supplier_scores, part_risk, kpis = build_analysis(DATA_DIR)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

part_risk.shape

(90, 17)

### 2. Summarize exposure and proposed actions

In [2]:
pd.DataFrame([
    {"measure": "Single-source parts", "value": kpis["single_source_parts"]},
    {"measure": "Potential stockouts", "value": kpis["potential_stockouts"]},
    {"measure": "Critical spend exposed", "value": f"${kpis['critical_spend_exposed']:,.0f}"},
    {"measure": "Parts requiring immediate mitigation", "value": int((part_risk["recommended_action"] != "Monitor").sum())},
])

,measure,value
0,Single-source parts,37
1,Potential stockouts,40
2,Critical spend exposed,"$33,438,854"
3,Parts requiring immediate mitigation,13


### 3. Review the action queue and supporting evidence

In [3]:
action_queue = part_risk[part_risk["recommended_action"] != "Monitor"][["part_id", "part_name", "category", "criticality", "supplier_count", "days_of_supply", "average_lead_time_days", "max_supplier_risk", "part_risk_score", "recommended_action"]].copy()
action_queue.head(20).round(1)

,part_id,part_name,category,criticality,supplier_count,days_of_supply,average_lead_time_days,max_supplier_risk,part_risk_score,recommended_action
0,PRT-0006,Avionic Assembly 006,Avionics,Critical,1,42.00,54.40,34.60,70.60,Expedite and allocate
1,PRT-0005,Avionic Assembly 005,Avionics,Critical,1,23.00,53.30,32.40,69.60,Expedite and allocate
2,PRT-0018,Interior Assembly 018,Interiors,Critical,1,37.00,51.20,32.10,69.40,Expedite and allocate
3,PRT-0031,Electrical Assembly 031,Electrical,Critical,1,36.00,64.20,27.90,67.60,Expedite and allocate
18,PRT-0042,Structure Assembly 042,Structures,Critical,1,62.00,53.10,34.20,50.40,Qualify a second source
19,PRT-0089,Propulsion Assembly 089,Propulsion,Critical,1,56.00,49.40,34.20,50.40,Qualify a second source
20,PRT-0074,Avionic Assembly 074,Avionics,Critical,2,32.00,52.30,39.00,47.60,Expedite and allocate
21,PRT-0086,Structure Assembly 086,Structures,Critical,3,21.00,52.30,34.60,45.60,Expedite and allocate
22,PRT-0049,Fastener Assembly 049,Fasteners,Critical,2,34.00,53.30,34.20,45.40,Expedite and allocate
23,PRT-0053,Electrical Assembly 053,Electrical,Critical,2,51.00,56.60,32.40,44.60,Expedite and allocate


### 4. Confirm that actions follow the stated rules

In [4]:
pd.DataFrame({
    "Expedite rule violations": [int(((part_risk["recommended_action"] == "Expedite and allocate") & ~(part_risk["stockout_risk"] & part_risk["critical_flag"])).sum())],
    "Dual-source rule violations": [int(((part_risk["recommended_action"] == "Qualify a second source") & ~(part_risk["single_source"] & part_risk["critical_flag"])).sum())],
})

,Expedite rule violations,Dual-source rule violations
0,0,0


## Results

The action queue prioritizes critical parts that may run out before replenishment, followed by critical single-source items. Rule checks confirm that proposed actions match documented conditions.

## Takeaways

- Evidence is preserved next to each recommendation.
- Buyers can distinguish immediate inventory intervention from longer-term sourcing work.
- Rules support consistent triage but do not replace planner judgment.
- Production use would add program impact, supplier capacity, mitigation cost, and action ownership.